# 06 - Analise Estatistica da Segmentacao Binarizada

Carrega a base analitica da segmentacao binarizada a partir do SQLite, filtrando a execucao escolhida em `config.toml`, persiste os resultados estatisticos no banco e exibe uma leitura inicial para comparacao entre estrategias e modelos.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis import (
    build_and_persist_analysis_segmentacao_binarizada_interacao_tag_estrategia,
    build_and_persist_analysis_segmentacao_binarizada_intervalo_confianca,
    build_and_persist_analysis_segmentacao_binarizada_resumo_estrategia,
    build_and_persist_analysis_segmentacao_binarizada_resumo_modelo_estrategia,
    build_and_persist_analysis_segmentacao_binarizada_resumo_tag,
    build_and_persist_analysis_segmentacao_binarizada_testes_estrategia,
    build_and_persist_analysis_segmentacao_binarizada_testes_tag_estrategia,
    build_binarized_metrics_dataframe,
)
from src.config import SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO
from src.repositories import (
    AnaliseSegmentacaoBinarizadaInteracaoTagEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaIntervaloConfiancaRepository,
    AnaliseSegmentacaoBinarizadaResumoEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaResumoModeloEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaResumoTagRepository,
    AnaliseSegmentacaoBinarizadaTesteEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaTesteTagEstrategiaRepository,
    ImagemRepository,
)


## Carregamento da base analitica em memoria

Le a avaliacao persistida no SQLite e monta a base analitica por `imagem + modelo + estrategia_binarizacao` apenas para a execucao escolhida em `config.toml`.


In [ ]:
imagem_repository = ImagemRepository()
# Docs: docs/decisoes-tecnicas/analise-da-segmentacao-binarizada-por-execucao-fixa.md
df_base = build_binarized_metrics_dataframe(
    imagem_repository.list(),
    execucao_escolhida=SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO,
)

print(f'Execucao escolhida para a analise binarizada: {SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO}')
print(f'Registros na base analitica: {len(df_base)}')
print(f'Imagens: {df_base["nome_arquivo"].nunique()}')
print(f'Modelos: {df_base["modelo"].nunique()}')
print(f'Execucoes presentes na base: {sorted(df_base["execucao"].unique().tolist())}')
print(f'Estrategias de binarizacao: {df_base["estrategia_binarizacao"].nunique()}')
print(f'Tags binarias disponiveis: {len([column for column in df_base.columns if column.startswith("tag_")])}')

df_base.head()


## Persistencia da camada analitica

Calcula os resumos descritivos, intervalos de confianca, comparacoes entre estrategias, testes por tag e interacoes `estrategia x tag`, persistindo tudo no SQLite. As analises entre execucoes ficam fora deste notebook por decisao tecnica.


In [ ]:
resumo_estrategia_repository = AnaliseSegmentacaoBinarizadaResumoEstrategiaRepository()
resumo_modelo_estrategia_repository = AnaliseSegmentacaoBinarizadaResumoModeloEstrategiaRepository()
resumo_tag_repository = AnaliseSegmentacaoBinarizadaResumoTagRepository()
intervalo_confianca_repository = AnaliseSegmentacaoBinarizadaIntervaloConfiancaRepository()
teste_estrategia_repository = AnaliseSegmentacaoBinarizadaTesteEstrategiaRepository()
teste_tag_estrategia_repository = AnaliseSegmentacaoBinarizadaTesteTagEstrategiaRepository()
interacao_tag_estrategia_repository = AnaliseSegmentacaoBinarizadaInteracaoTagEstrategiaRepository()

df_resumo_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_resumo_estrategia(
    df_base=df_base,
    repository=resumo_estrategia_repository,
)
df_resumo_modelo_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_resumo_modelo_estrategia(
    df_base=df_base,
    repository=resumo_modelo_estrategia_repository,
)
df_resumo_tag, _ = build_and_persist_analysis_segmentacao_binarizada_resumo_tag(
    df_base=df_base,
    repository=resumo_tag_repository,
)
df_intervalo_confianca, _ = build_and_persist_analysis_segmentacao_binarizada_intervalo_confianca(
    df_base=df_base,
    repository=intervalo_confianca_repository,
)
df_testes_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_testes_estrategia(
    df_base=df_base,
    repository=teste_estrategia_repository,
)
df_testes_tag_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_testes_tag_estrategia(
    df_base=df_base,
    repository=teste_tag_estrategia_repository,
)
df_interacoes_tag_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_interacao_tag_estrategia(
    df_base=df_base,
    repository=interacao_tag_estrategia_repository,
)

print(f'Registros no resumo por estrategia: {len(df_resumo_estrategia)}')
print(f'Registros no resumo por modelo + estrategia: {len(df_resumo_modelo_estrategia)}')
print(f'Registros no resumo por tag: {len(df_resumo_tag)}')
print(f'Registros nos intervalos de confianca: {len(df_intervalo_confianca)}')
print(f'Registros nos testes entre estrategias: {len(df_testes_estrategia)}')
print(f'Registros nos testes por tag: {len(df_testes_tag_estrategia)}')
print(f'Registros nas interacoes estrategia x tag: {len(df_interacoes_tag_estrategia)}')


## Leitura inicial dos resultados

Mostra uma visao compacta das tabelas persistidas para orientar a etapa visual do notebook 07.


In [ ]:
pd.pivot_table(
    df_resumo_estrategia,
    index='estrategia_binarizacao',
    columns='metric_name',
    values=['mean', 'median'],
)


In [ ]:
df_resumo_modelo_estrategia.sort_values(['metric_name', 'modelo', 'mean'], ascending=[True, True, False]).head(24)


In [ ]:
df_intervalo_confianca.sort_values(['metric_name', 'statistic_name', 'modelo', 'estrategia_binarizacao']).head(24)


In [ ]:
if df_testes_estrategia.empty:
    print('Nao ha estrategias suficientes no banco atual para executar comparacoes estatisticas.')
else:
    df_testes_estrategia.sort_values(['metric_name', 'modelo_origem', 'comparison_scope', 'p_value_adjusted']).head(30)


In [ ]:
df_testes_tag_estrategia.sort_values(['metric_name', 'estrategia_binarizacao', 'tag_name', 'comparison_scope', 'p_value_adjusted']).head(30)


In [ ]:
df_interacoes_tag_estrategia.sort_values(['metric_name', 'estrategia_binarizacao', 'adjusted_delta_mean']).head(30)
